# Multimodal Fine-Tuning: Handwritten LaTeX OCR Expert

This tutorial demonstrates how to fine-tune a state-of-the-art vision-language model, **Qwen2-VL 2B Instruct**, for document understanding and LaTeX formula transcription. We use **Unsloth** for memory-efficient QLoRA training and the **unsloth/LaTeX_OCR** dataset from the Hugging Face hub.

## 1. Install Dependencies & Setup Environment
We install the required libraries (`unsloth`, `trl`, `peft`, `bitsandbytes`, `transformers`) to run our training.

In [ ]:
import os
import torch
from datasets import load_dataset
from unsloth import FastVisionModel
from trl import SFTTrainer, SFTConfig
from unsloth.trainer import UnslothVisionDataCollator

compute_dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8 else torch.float16
print('Environment initialized successfully.')

## 2. Load Model and Processor (FastVisionModel)
We load `unsloth/Qwen2-VL-2B-Instruct` in 4-bit precision to fit within the 16GB VRAM constraint, and enable gradient checkpointing for lower VRAM footprints.

In [ ]:
MODEL_ID = 'unsloth/Qwen2-VL-2B-Instruct'

model, tokenizer = FastVisionModel.from_pretrained(
    model_name=MODEL_ID,
    load_in_4bit=True,
    use_gradient_checkpointing='unsloth',
)

model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=True,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    random_state=3407,
)
print('Qwen2-VL and PEFT adapters loaded successfully.')

## 3. Load and Format Dataset (LaTeX_OCR)
We load `unsloth/LaTeX_OCR` directly from Hugging Face and map the examples into a conversational format suitable for vision-language models.

In [ ]:
dataset = load_dataset('unsloth/LaTeX_OCR', split='train')

# Select a subset to train and validate quickly
shuffled_dataset = dataset.shuffle(seed=42)
train_ds = shuffled_dataset.select(range(500))
eval_ds = shuffled_dataset.select(range(500, 550))

def convert_to_conversation(sample):
    conversation = [
        {
            'role': 'user',
            'content': [
                {'type': 'text', 'text': 'Write the LaTeX representation for this image.'},
                {'type': 'image'}
            ]
        },
        {
            'role': 'assistant',
            'content': [
                {'type': 'text', 'text': sample['text']}
            ]
        },
    ]
    return {
        'messages': conversation,
        'images': [sample['image']]
    }

train_mapped = train_ds.map(convert_to_conversation, remove_columns=train_ds.column_names)
eval_mapped = eval_ds.map(convert_to_conversation, remove_columns=eval_ds.column_names)
print('Dataset sample mapped successfully.')

## 4. Run SFT Trainer
We configure the Hugging Face `trl` SFTTrainer with Unsloth's optimized vision data collator. We set `remove_unused_columns = False` to prevent losing the image data during training.

In [ ]:
training_args = SFTConfig(
    output_dir='qwen2-vl-latex',
    dataset_text_field='text',
    max_seq_length=512,
    remove_unused_columns=False,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_steps=5,
    max_steps=30,
    bf16=(compute_dtype == torch.bfloat16),
    logging_steps=5,
    eval_strategy='steps',
    eval_steps=10,
    save_steps=10,
    report_to='none',
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    data_collator=UnslothVisionDataCollator(model, tokenizer),
    train_dataset=train_mapped,
    eval_dataset=eval_mapped,
    args=training_args,
)

# Disable KV cache during training to save memory
model.config.use_cache = False

trainer.train()

# Save the fine-tuned adapter
model.save_pretrained('qwen2-vl-latex-adapter')
tokenizer.save_pretrained('qwen2-vl-latex-adapter')
print('Vision adapter successfully saved!')

## 5. Evaluation and Inference
We switch the model to inference mode and generate a LaTeX translation for a validation image.

In [ ]:
FastVisionModel.for_inference(model)

sample = eval_ds[0]
image = sample['image']
expected_latex = sample['text']

messages = [
    {
        'role': 'user',
        'content': [
            {'type': 'text', 'text': 'Write the LaTeX representation for this image.'},
            {'type': 'image', 'image': image}
        ]
    }
]

input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
inputs = tokenizer(
    image,
    input_text,
    add_special_tokens=False,
    return_tensors='pt'
).to('cuda')

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=128)

generated_text = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

print('--- Expected LaTeX ---')
print(expected_latex)
print('\n--- Generated LaTeX ---')
print(generated_text)